# Week 34, Day 3: Tuning curves: fitting, uncertainty and selectivity

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w034/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: A von Mises fit

`counts` holds Poisson responses of a neuron to 8 orientations, 10 trials each. Complete `fit(y)`, which takes the 8 mean responses: for every (θ_p, κ) on the grid, the basis G is precomputed; solve for b and A by least squares and return the (b, A, θ_p, κ) with the smallest squared error.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(62)
ori = np.arange(0, 180, 22.5)

def vm(th, b, A, tp, k):
    return b + A * np.exp(k * (np.cos(2 * np.deg2rad(th - tp)) - 1))

counts = r.poisson(vm(ori, 2, 18, 70, 2.0)[:, None] * np.ones(10))      # orientations x trials
TP, K = np.arange(0, 180, 1.0), np.arange(0.25, 6.01, 0.25)
G = np.exp(K[None, :, None] * (np.cos(2 * np.deg2rad(ori[None, None, :] - TP[:, None, None])) - 1))   # tp x k x ori

def fit(y):
    n = len(y)
    sg, sgg, sy, sgy = G.sum(-1), (G * G).sum(-1), y.sum(), (G * y).sum(-1)
    det = n * sgg - sg * sg
    A = (n * sgy - sg * sy) / det
    b = (sy - A * sg) / n
    err = ...
    i = np.unravel_index(np.argmin(err), err.shape)
    return b[i], A[i], TP[i[0]], K[i[1]]

params = fit(counts.mean(1))
print("baseline {:.1f}, amplitude {:.1f}, preferred {:.0f} deg, kappa {:.2f}".format(*params))
th = np.linspace(0, 180, 181)
plt.plot(ori, counts.mean(1), "o"); plt.plot(th, vm(th, *params)); plt.xlabel("orientation (deg)"); plt.ylabel("spikes per trial"); plt.show()

In [ ]:
import numpy as np
def test_fit():
    "the fit recovers the preferred orientation within 5 degrees and a positive amplitude"
    b, A, tp, k = params
    assert abs(tp - 70) < 5 and A > 10

test_fit()
print("All checks passed")

### Exercise: A confidence interval by resampling

Write `boot_ci(counts, n_boot=200)`: resample trials with replacement within each orientation, refit, and return the 2.5th and 97.5th percentiles of the fitted preferred orientations. Apply it to all 10 trials (`ci`) and to the first 3 trials only (`ci3`).

In [ ]:
def boot_ci(c, n_boot=200):
    pref = []
    for _ in range(n_boot):
        idx = r.integers(0, c.shape[1], c.shape)
        y = ...
        pref.append(fit(y)[2])
    return np.percentile(pref, [2.5, 97.5])

ci = boot_ci(counts)
ci3 = boot_ci(counts[:, :3])
print(f"10 trials: preferred orientation 95% interval {ci[0]:.0f} to {ci[1]:.0f} deg;  3 trials: {ci3[0]:.0f} to {ci3[1]:.0f} deg")

In [ ]:
import numpy as np
def test_ci():
    "the 95% interval contains the true 70 degrees and is narrower than 20 degrees"
    assert ci[0] <= 70 <= ci[1] and ci[1] - ci[0] < 20

def test_fewer():
    "with only 3 trials per orientation the interval is wider"
    assert (ci3[1] - ci3[0]) > (ci[1] - ci[0])

test_ci()
test_fewer()
print("All checks passed")

### Exercise: Tuning from nothing

Write `osi(c)` for a counts array (orientations x trials). Measure its average over 300 simulated untuned neurons (2 spikes per trial at every orientation) with 5 and with 50 trials (`bias[n]`). Then write `shuffle_p(c)`: the fraction of 500 label shuffles (permute all trials across orientations) whose OSI is at least the real one; apply it to `counts` and to one untuned neuron with 5 trials.

In [ ]:
def osi(c):
    m = c.mean(1)
    return ...

bias = {n: np.mean([osi(r.poisson(2.0, (8, n))) for _ in range(300)]) for n in (5, 50)}
print(f"untuned neuron, mean OSI: {bias[5]:.3f} with 5 trials, {bias[50]:.3f} with 50 trials;  tuned neuron OSI {osi(counts):.2f}")

def shuffle_p(c, n_shuf=500):
    real, flat = osi(c), c.ravel()
    null = [osi(r.permutation(flat).reshape(c.shape)) for _ in range(n_shuf)]
    return ...

untuned = r.poisson(2.0, (8, 5))
p_tuned, p_untuned = shuffle_p(counts), shuffle_p(untuned)
print(f"shuffle p-values: tuned neuron {p_tuned:.3f} (OSI {osi(counts):.2f}), untuned neuron {p_untuned:.3f} (OSI {osi(untuned):.2f})")

In [ ]:
import numpy as np
def test_bias():
    "an untuned neuron's mean OSI is clearly above 0 with 5 trials and smaller with 50"
    assert bias[5] > 0.07 and bias[50] < 0.6 * bias[5]

def test_shuffle():
    "the tuned neuron's OSI exceeds 95% of its shuffles, the untuned one's does not"
    assert p_tuned < 0.05 and p_untuned > 0.05

test_bias()
test_shuffle()
print("All checks passed")